In [1]:
from pathlib import Path
import re
import csv
from collections import Counter
from datetime import datetime

# Optional: jika pandas tersedia, kita gunakan untuk tampilan tabel yang lebih rapi
try:
    import pandas as pd
except ImportError:
    pd = None

DATA_PATH = Path("../data/hasil_generate/data_mahasiswaaa.txt")

print("Apakah file ada?", DATA_PATH.exists())
print("Lokasi file:", DATA_PATH.resolve())

Apakah file ada? True
Lokasi file: /Users/mm/PycharmProjects/PemrogSem2/data/data_mahasiswaaa.txt


## Studi Kasus 1 — Membaca file `.txt`

Pada bagian ini kita menerapkan materi **file handling**:
- `open()`
- mode baca (`'r'`)
- context manager `with`
- metode `.read()` dan `.readlines()`

In [2]:
# Membaca seluruh isi file sebagai satu string
with open(DATA_PATH, "r", encoding="utf-8") as file:
    isi_file = file.read()

print("=== Isi file mentah ===")
print(isi_file)

=== Isi file mentah ===
Andi, Teknik Informatika, 08224541001, 12-08-2000, 3130600018
Budi, 23-01-2001, Sains Data Terapan, 332450001, @budi_sdt_24
Citra, 05/05/1999, 2317500023, citra@pens.ac.id
Dewi, 3321600018, sains data terapan, 17-11-2000, dewi@pemrograman.id



In [3]:
# Membaca file per baris
with open(DATA_PATH, "r", encoding="utf-8") as file:
    lines = file.readlines()

print("=== Daftar baris ===")
for i, line in enumerate(lines, start=1):
    print(f"Baris {i}: {line.strip()}")

=== Daftar baris ===
Baris 1: Andi, Teknik Informatika, 08224541001, 12-08-2000, 3130600018
Baris 2: Budi, 23-01-2001, Sains Data Terapan, 332450001, @budi_sdt_24
Baris 3: Citra, 05/05/1999, 2317500023, citra@pens.ac.id
Baris 4: Dewi, 3321600018, sains data terapan, 17-11-2000, dewi@pemrograman.id


## Studi Kasus 2 — Analisis pola data dengan Regex

Dari file terlihat bahwa data mahasiswa **tidak konsisten**:
- urutan kolom berbeda-beda,
- format tanggal berbeda (`12-08-2000` dan `05/05/1999`),
- ada data email, username sosial media, nomor telepon, NRP/NIM, dan program studi.

Kita akan membuat regex untuk mendeteksi:
- **tanggal lahir**
- **email**
- **username sosial media**
- **angka/NRP/nomor telepon**

In [4]:
# Pola regex
tanggal_pattern = re.compile(r"\b\d{2}[-/]\d{2}[-/]\d{4}\b")
email_pattern = re.compile(r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b")
sosmed_pattern = re.compile(r"^@\w+$")
angka_pattern = re.compile(r"^\d+$")

for i, line in enumerate(lines, start=1):
    teks = line.strip()

    tanggal = tanggal_pattern.findall(teks)
    email = email_pattern.findall(teks)
    sosmed = [token.strip() for token in teks.split(",") if sosmed_pattern.fullmatch(token.strip())]
    angka = [token.strip() for token in teks.split(",") if angka_pattern.fullmatch(token.strip())]

    print(f"--- Baris {i} ---")
    print("Teks   :", teks)
    print("Tanggal:", tanggal)
    print("Email  :", email)
    print("Sosmed :", sosmed)
    print("Angka  :", angka)
    print()

--- Baris 1 ---
Teks   : Andi, Teknik Informatika, 08224541001, 12-08-2000, 3130600018
Tanggal: ['12-08-2000']
Email  : []
Sosmed : []
Angka  : ['08224541001', '3130600018']

--- Baris 2 ---
Teks   : Budi, 23-01-2001, Sains Data Terapan, 332450001, @budi_sdt_24
Tanggal: ['23-01-2001']
Email  : []
Sosmed : ['@budi_sdt_24']
Angka  : ['332450001']

--- Baris 3 ---
Teks   : Citra, 05/05/1999, 2317500023, citra@pens.ac.id
Tanggal: ['05/05/1999']
Email  : ['citra@pens.ac.id']
Sosmed : []
Angka  : ['2317500023']

--- Baris 4 ---
Teks   : Dewi, 3321600018, sains data terapan, 17-11-2000, dewi@pemrograman.id
Tanggal: ['17-11-2000']
Email  : ['dewi@pemrograman.id']
Sosmed : []
Angka  : ['3321600018']



## Studi Kasus 3 — Parsing data mahasiswa menjadi struktur yang rapi

Aturan sederhana yang kita pakai:
- token pertama = **nama**
- jika token cocok regex tanggal → **tanggal_lahir**
- jika token cocok regex email → **email**
- jika token cocok regex username → **username**
- jika token berupa angka:
  - jika diawali `0` → **no_hp**
  - selain itu → **nrp**
- token teks lain → **prodi**

> Catatan: aturan ini adalah **heuristik** sederhana untuk latihan regex dan pembersihan data.

In [5]:
def normalisasi_tanggal(tanggal_str):
    if not tanggal_str:
        return None

    for fmt in ("%d-%m-%Y", "%d/%m/%Y"):
        try:
            return datetime.strptime(tanggal_str, fmt).strftime("%Y-%m-%d")
        except ValueError:
            pass
    return None


def parse_line(line):
    tokens = [token.strip() for token in line.strip().split(",")]
    record = {
        "nama": None,
        "prodi": None,
        "no_hp": None,
        "tanggal_lahir": None,
        "tanggal_lahir_iso": None,
        "nrp": None,
        "email": None,
        "username": None,
    }

    if tokens:
        record["nama"] = tokens[0]

    for token in tokens[1:]:
        if tanggal_pattern.fullmatch(token):
            record["tanggal_lahir"] = token
            record["tanggal_lahir_iso"] = normalisasi_tanggal(token)
        elif email_pattern.fullmatch(token):
            record["email"] = token
        elif sosmed_pattern.fullmatch(token):
            record["username"] = token
        elif angka_pattern.fullmatch(token):
            if token.startswith("0"):
                record["no_hp"] = token
            else:
                # kalau no_hp belum ada dan panjang angka besar, bisa dianggap no_hp
                if record["nrp"] is None:
                    record["nrp"] = token
                else:
                    # fallback jika ternyata ada angka kedua
                    if record["no_hp"] is None:
                        record["no_hp"] = token
        else:
            record["prodi"] = token.title()

    return record


parsed_data = [parse_line(line) for line in lines]

for item in parsed_data:
    print(item)

{'nama': 'Andi', 'prodi': 'Teknik Informatika', 'no_hp': '08224541001', 'tanggal_lahir': '12-08-2000', 'tanggal_lahir_iso': '2000-08-12', 'nrp': '3130600018', 'email': None, 'username': None}
{'nama': 'Budi', 'prodi': 'Sains Data Terapan', 'no_hp': None, 'tanggal_lahir': '23-01-2001', 'tanggal_lahir_iso': '2001-01-23', 'nrp': '332450001', 'email': None, 'username': '@budi_sdt_24'}
{'nama': 'Citra', 'prodi': None, 'no_hp': None, 'tanggal_lahir': '05/05/1999', 'tanggal_lahir_iso': '1999-05-05', 'nrp': '2317500023', 'email': 'citra@pens.ac.id', 'username': None}
{'nama': 'Dewi', 'prodi': 'Sains Data Terapan', 'no_hp': None, 'tanggal_lahir': '17-11-2000', 'tanggal_lahir_iso': '2000-11-17', 'nrp': '3321600018', 'email': 'dewi@pemrograman.id', 'username': None}


In [6]:
if pd is not None:
    df = pd.DataFrame(parsed_data)
    display(df)
else:
    print(parsed_data)

,nama,prodi,no_hp,tanggal_lahir,tanggal_lahir_iso,nrp,email,username
0,Andi,Teknik Informatika,08224541001,12-08-2000,2000-08-12,3130600018,NaN,NaN
1,Budi,Sains Data Terapan,NaN,23-01-2001,2001-01-23,332450001,NaN,@budi_sdt_24
2,Citra,NaN,NaN,05/05/1999,1999-05-05,2317500023,citra@pens.ac.id,NaN
3,Dewi,Sains Data Terapan,NaN,17-11-2000,2000-11-17,3321600018,dewi@pemrograman.id,NaN


## Studi Kasus 4 — Validasi data menggunakan regex

Sekarang kita cek kualitas data:
- apakah email valid?
- apakah nomor HP sesuai pola sederhana Indonesia (`0` di depan, panjang 10–13 digit)?
- apakah NRP berisi angka saja?
- apakah tanggal berhasil dinormalisasi?

In [7]:
def valid_email(email):
    if not email:
        return False
    return bool(email_pattern.fullmatch(email))

def valid_no_hp(no_hp):
    if not no_hp:
        return False
    return bool(re.fullmatch(r"0\d{9,12}", no_hp))

def valid_nrp(nrp):
    if not nrp:
        return False
    return bool(re.fullmatch(r"\d{9,10}", nrp))

def valid_tanggal_iso(tanggal_iso):
    return tanggal_iso is not None

hasil_validasi = []
for item in parsed_data:
    hasil_validasi.append({
        "nama": item["nama"],
        "email_valid": valid_email(item["email"]),
        "no_hp_valid": valid_no_hp(item["no_hp"]),
        "nrp_valid": valid_nrp(item["nrp"]),
        "tanggal_valid": valid_tanggal_iso(item["tanggal_lahir_iso"]),
    })

if pd is not None:
    df_validasi = pd.DataFrame(hasil_validasi)
    display(df_validasi)
else:
    print(hasil_validasi)

,nama,email_valid,no_hp_valid,nrp_valid,tanggal_valid
0,Andi,False,True,True,True
1,Budi,False,False,True,True
2,Citra,True,False,True,True
3,Dewi,True,False,True,True


## Studi Kasus 5 — Ringkasan data mahasiswa

Dari data yang sudah dibersihkan, kita bisa membuat ringkasan sederhana:
- jumlah mahasiswa
- jumlah yang punya email
- jumlah yang punya username
- jumlah per prodi

In [8]:
jumlah_mahasiswa = len(parsed_data)
jumlah_email = sum(1 for item in parsed_data if item["email"])
jumlah_username = sum(1 for item in parsed_data if item["username"])
jumlah_no_hp = sum(1 for item in parsed_data if item["no_hp"])
jumlah_prodi = Counter(item["prodi"] for item in parsed_data if item["prodi"])

print("Jumlah mahasiswa       :", jumlah_mahasiswa)
print("Memiliki email         :", jumlah_email)
print("Memiliki username      :", jumlah_username)
print("Memiliki nomor HP      :", jumlah_no_hp)
print("Distribusi program studi:")
for prodi, jumlah in jumlah_prodi.items():
    print(f"- {prodi}: {jumlah}")

Jumlah mahasiswa       : 4
Memiliki email         : 2
Memiliki username      : 1
Memiliki nomor HP      : 1
Distribusi program studi:
- Teknik Informatika: 1
- Sains Data Terapan: 2


## Studi Kasus 6 — Menyimpan hasil bersih ke file CSV

Setelah data diparsing dan dinormalisasi, hasilnya bisa disimpan ke file `.csv`.
Ini menggabungkan materi:
- **file handling**
- **library `csv`**
- proses pembersihan data

In [9]:
output_csv = Path("../data/hasil_generate/hasil_bersih_mahasiswa.csv")

fieldnames = [
    "nama",
    "prodi",
    "no_hp",
    "tanggal_lahir",
    "tanggal_lahir_iso",
    "nrp",
    "email",
    "username",
]

with open(output_csv, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(parsed_data)

print(f"File CSV berhasil dibuat: {output_csv.resolve()}")

File CSV berhasil dibuat: /Users/mm/PycharmProjects/PemrogSem2/study/hasil_bersih_mahasiswa.csv


In [10]:
# Menampilkan isi CSV hasil akhir
with open("../data/hasil_generate/hasil_bersih_mahasiswa.csv", "r", encoding="utf-8") as file:
    print(file.read())

nama,prodi,no_hp,tanggal_lahir,tanggal_lahir_iso,nrp,email,username
Andi,Teknik Informatika,08224541001,12-08-2000,2000-08-12,3130600018,,
Budi,Sains Data Terapan,,23-01-2001,2001-01-23,332450001,,@budi_sdt_24
Citra,,,05/05/1999,1999-05-05,2317500023,citra@pens.ac.id,
Dewi,Sains Data Terapan,,17-11-2000,2000-11-17,3321600018,dewi@pemrograman.id,



## Studi Kasus 7 — Membuat fungsi reusable seperti mini-module

Sesuai materi **library/modul**, kita juga bisa membungkus logika parsing ke dalam fungsi
agar dapat digunakan ulang untuk file lain yang formatnya mirip.

In [11]:
def parse_file_mahasiswa(path_file):
    path_file = Path(path_file)
    hasil = []

    with open(path_file, "r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                hasil.append(parse_line(line))

    return hasil

hasil_dari_fungsi = parse_file_mahasiswa(DATA_PATH)

if pd is not None:
    display(pd.DataFrame(hasil_dari_fungsi))
else:
    print(hasil_dari_fungsi)

,nama,prodi,no_hp,tanggal_lahir,tanggal_lahir_iso,nrp,email,username
0,Andi,Teknik Informatika,08224541001,12-08-2000,2000-08-12,3130600018,NaN,NaN
1,Budi,Sains Data Terapan,NaN,23-01-2001,2001-01-23,332450001,NaN,@budi_sdt_24
2,Citra,NaN,NaN,05/05/1999,1999-05-05,2317500023,citra@pens.ac.id,NaN
3,Dewi,Sains Data Terapan,NaN,17-11-2000,2000-11-17,3321600018,dewi@pemrograman.id,NaN


In [13]:
import re
import pandas as pd
import matplotlib.pyplot as plt

#Baca file dan siapkan data mentah
with open("data_mahasiswa (1).txt", "r", encoding="utf-8") as f:
    lines = [line.strip() for line in f if line.strip()]

print("Isi data mentah:")
for i, line in enumerate(lines, start=1):
    print(f"{i}. {line}")

#Fungsi validasi yang lebih ketat
# Hanya domain kampus tertentu yang diterima
ALLOWED_EMAIL_DOMAINS = ["pens.ac.id"]

def is_email(value):
    return re.fullmatch(r"[A-Za-z0-9._%+-]+@([A-Za-z0-9.-]+\.[A-Za-z]{2,})", value) is not None

def is_allowed_campus_email(value):
    if not is_email(value):
        return False
    domain = value.split("@")[-1].lower()
    return domain in ALLOWED_EMAIL_DOMAINS

# Format tanggal: dd-mm-yyyy atau dd/mm/yyyy
def is_date(value):
    return re.fullmatch(r"(0[1-9]|[12][0-9]|3[01])[-/](0[1-9]|1[0-2])[-/](19\d{2}|20\d{2})", value) is not None

# Nomor telepon Indonesia: 08xxxxxxxxxx (10 s.d. 13 digit)
def is_phone(value):
    return re.fullmatch(r"08\d{8,11}", value) is not None

# NIM/NRP lebih ketat: hanya angka, panjang 9 s.d. 10 digit
def is_nim_nrp(value):
    return re.fullmatch(r"\d{9,10}", value) is not None

# Username sosial/media handle
def is_username(value):
    return re.fullmatch(r"@[A-Za-z0-9_\.]+", value) is not None

# Nama dan program studi
def looks_like_name(value):
    return re.fullmatch(r"[A-Za-z ]{2,}", value) is not None

def looks_like_prodi(value):
    return re.fullmatch(r"[A-Za-z ]{3,}", value) is not None

#Parsing data per baris dan klasifikasi token
parsed_data = []

for line in lines:
    parts = [p.strip() for p in line.split(",")]

    row = {
        "nama": None,
        "prodi": None,
        "telepon": None,
        "tanggal_lahir": None,
        "nim_nrp": None,
        "email": None,
        "username": None,
        "baris_asli": line
    }

    unknown_parts = []

    for part in parts:
        part_lower = part.lower()

        if looks_like_name(part) and row["nama"] is None:
            row["nama"] = part

        elif is_phone(part) and row["telepon"] is None:
            row["telepon"] = part

        elif is_date(part) and row["tanggal_lahir"] is None:
            row["tanggal_lahir"] = part

        elif is_nim_nrp(part) and row["nim_nrp"] is None:
            row["nim_nrp"] = part

        elif is_email(part) and row["email"] is None:
            row["email"] = part

        elif is_username(part) and row["username"] is None:
            row["username"] = part

        elif looks_like_prodi(part) and row["prodi"] is None:
            row["prodi"] = part

        else:
            unknown_parts.append(part)

    row["unknown_parts"] = unknown_parts
    parsed_data.append(row)

df = pd.DataFrame(parsed_data)
df

#Validasi Domain Email
def validate_row(row):
    catatan = []

    if not row["nama"]:
        catatan.append("nama_tidak_terdeteksi")

    if row["email"]:
        if not is_allowed_campus_email(row["email"]):
            catatan.append("email_bukan_domain_kampus")
    else:
        catatan.append("email_tidak_ada")

    if row["telepon"]:
        if not is_phone(row["telepon"]):
            catatan.append("telepon_tidak_valid")
    else:
        catatan.append("telepon_tidak_ada")

    if row["nim_nrp"]:
        if not is_nim_nrp(row["nim_nrp"]):
            catatan.append("nim_nrp_tidak_valid")
    else:
        catatan.append("nim_nrp_tidak_ada")

    if row["tanggal_lahir"]:
        if not is_date(row["tanggal_lahir"]):
            catatan.append("tanggal_tidak_valid")
    else:
        catatan.append("tanggal_tidak_ada")

    status = "valid" if len(catatan) == 0 else "perlu_cek"
    return pd.Series([status, "; ".join(catatan)])

df[["status_data", "catatan_validasi"]] = df.apply(validate_row, axis=1)
df



FileNotFoundError: [Errno 2] No such file or directory: 'data_mahasiswa (1).txt'

## Kesimpulan

Melalui notebook ini, kita sudah mempraktikkan:
- **Library**: memakai `re`, `csv`, `pathlib`, `collections`
- **File Handling**: membaca file txt dan menulis CSV
- **Regex**: mengekstrak pola tanggal, email, username, dan angka

Contoh ini cocok dijadikan dasar untuk tugas praktikum atau pengembangan studi kasus data teks yang tidak rapi.